## GRG 356T Spatial Data Science and Maps
### The University of Texas at Austin, Department of Geography and the Environment
Developed by Yuhao Kang (instructor), Xihan Yao and Brody Manquen. Part of *Responsible GeoAI for Civic Futures*, supported by the Mozilla Foundation.


# Lab 10 — Generative AI in Geography and Cartography

Large language models (LLMs) such as ChatGPT can now do real cartographic work: describe a
painting, pick a colour scheme, write the file that styles a web map. They will also answer
a question about any place on Earth, and the answer always sounds confident. This lab looks
at both sides.

- **Part A -- AI as a cartographer.** You give a model a picture; it designs a matching
  style for a map of the UT campus, which you open in Mapbox Studio. Then the model judges
  its own map. This part follows **CartoAgent**, developed in Prof. Kang's GISense Lab.
- **Part B -- Can we trust what AI knows about places?** Texas counties range from
  Harris (5 million people) to Loving (48). One possible reason a model could know small places
  less well is that far less text is written about them -- but this lab never sees what a model
  was trained on, so treat that as a hypothesis to test, not a fact. You ask LLMs questions about all 254 Texas counties and check the
  answers against official data. You also compare an ordinary model released in 2024 with a
  current one: does a newer model fix the problem?

The theme of the whole lab is **data quality**: what generative AI is good for, and when you
should check its work.

**How these readings map onto the lab**

Both studies below tested models from 2023-24. You will run the neighbour test on a model
from 2024 and a model released in 2025.

| Reading | Section here | What they found |
|---|---|---|
| Renshaw et al. (2025) | **B2** -- the neighbor test | ChatGPT and Gemini Pro named neighbouring counties correctly 49-79% of the time, and did worse in rural areas. |
| Kim et al. (2024) | background for the whole of Part B | Of 3,108 US counties, only 515 (16.6%) got an answer specific to that county. Counties with lower density and lower income were less likely to get one. |

That leaves one question neither paper could answer, and it is the one this lab is really
about: **two years and one model generation later, is the gap between big places and small
places any smaller?** Section B2 prints the numbers you need; do not assume the answer.

**Readings**

- CartoAgent: a multimodal large language model-powered multi-agent cartographic framework
  for map style transfer and evaluation. *International Journal of Geographical Information
  Science* (2025). https://doi.org/10.1080/13658816.2025.2507844 -- code:
  https://github.com/GISense/CartoAgent
- Kim, J., Lee, J., Jang, K. M., & Lourentzou, I. (2024). Exploring the limitations in how
  ChatGPT introduces environmental justice issues in the United States: A case study of
  3,108 counties. *Telematics and Informatics*, 86, 102085.
  https://doi.org/10.1016/j.tele.2023.102085
- Renshaw, A., Lourentzou, I., Lee, J., Crawford, T., & Kim, J. (2025). Comparing the spatial
  querying capacity of large language models: OpenAI's ChatGPT and Google's Gemini Pro.
  *The Professional Geographer*, 77(2), 186-198.
  https://doi.org/10.1080/00330124.2024.2434455

## Lab Sections

0. Connect to the OpenAI API

**Part A -- AI as a cartographer**

- A1. A map style is a text file
- A2. The four prompts
- A3. From a photograph to a map style
- A4. See your map in Mapbox Studio
- A5. Let the model judge its own map
- Exercise 1 -- a map in the style of an artwork

**Part B -- Can we trust what AI knows about places?**

- B1. Asking an LLM about a place
- B2. The neighbor test -- does the LLM know which counties are next to each other?
- B3. Does a better prompt fix it?
- Exercises 2-3

### Where to run this lab

Anywhere: Google Colab, your own laptop, or iGuide. The Setup cell below installs
everything this lab needs.

The lab reads files from the `data` folder that came with it. In Colab, upload the whole
`data` **folder** (not the individual files) next to this notebook.

You also need two free accounts: an **OpenAI API key** (Section 0) and a **Mapbox** account
for Part A (section A4 explains how).

## Setup

Run the cell below **first**. It checks that every package this lab needs is
installed, and installs anything that is missing. If it installs something,
restart the kernel and run it again.

In [ ]:
# =====================================================================
# Lab 10 -- Setup: check that every package this lab needs is installed
# AND new enough. Run this cell FIRST. It is safe to re-run.
# =====================================================================

import importlib.metadata as md
import subprocess, sys

# pip package name -> minimum version this lab was written against
REQUIRED = {
    "openai":     "1.40",
    "geopandas":  "1.0",
    "pandas":     "2.0",
    "numpy":      "1.26",
    "matplotlib": "3.8",
    "pillow":     "10.0",
}


def version_tuple(text):
    """'1.10.2' -> (1, 10, 2), so 1.10 correctly counts as newer than 1.9."""
    parts = []
    for piece in text.split(".")[:3]:
        digits = "".join(ch for ch in piece if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


problems = []
for package, minimum in REQUIRED.items():
    try:
        installed = md.version(package)
    except md.PackageNotFoundError:
        problems.append((package, minimum, "not installed"))
        continue
    if version_tuple(installed) < version_tuple(minimum):
        problems.append((package, minimum, "you have " + installed))

if problems:
    print("Installing or upgrading " + str(len(problems)) + " package(s):")
    for package, minimum, why in problems:
        print("   " + package + "  (needs >=" + minimum + ", " + why + ")")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                           *[p + ">=" + m for p, m, _ in problems]])
    print("\nInstall finished.")
    if "google.colab" in sys.modules:
        print(">>> Now use the menu:  Runtime > Restart session,  then run this cell again. <<<")
    else:
        print(">>> RESTART THE KERNEL now (Kernel > Restart), then run this cell again to confirm. <<<")
else:
    print("All " + str(len(REQUIRED)) + " required packages are present and new enough.")
    for package in REQUIRED:
        print("   " + package.ljust(18) + md.version(package))

### Imports

Every package this lab uses is imported here, once, with a note on what it is for.

In [ ]:
# =====================================================================
# Lab 10 -- every import this notebook needs, in one place.
# Run this cell once, right after the Setup cell above. If a later cell
# raises NameError, come back and re-run this one.
# =====================================================================

# --- standard library ---
import getpass                                # type the class API key in without it showing on screen
import hashlib                                # give each saved answer a short, stable filename
import json                                   # save each answer in llm_cache/ and read it back
import pathlib                                # build the llm_cache folder
import re                                     # pull county names out of the model's reply
import threading                              # count new questions safely while several run at once
import base64                                 # turn an image into text so it can be sent to the model
import io                                    # shrink an image in memory before sending it
from concurrent.futures import ThreadPoolExecutor, as_completed   # ask several questions at the same time

# --- the LLM ---
from openai import OpenAI                     # sends each question to the model and returns its reply

# --- data and maps ---
import numpy as np                            # shuffles the county order in the neighbour test
import pandas as pd                           # one table row per county: answer, score, population
import geopandas as gpd                       # Texas county polygons for the score map
import matplotlib.pyplot as plt               # the map, the scatter plot and the bar charts

# --- images (Part A) ---
from PIL import Image                         # open, shrink and show the pictures and map screenshots
from IPython.display import display           # show a picture inside the notebook

# Section 0 -- Connect to the OpenAI API

We use OpenAI's models -- the same ones as ChatGPT -- called from Python instead of typed
into a chat window. You need an API key:

- **In a class:** use the key your instructor gives you. A UT AI Studio (Portkey) key also
  works; the cell below recognises it automatically.
- **Everyone else:** create your own OpenAI key at https://platform.openai.com/api-keys
  (it starts with `sk-`). A full run of this lab costs about $1.20.

Paste the key when the cell below asks for it. The key is never saved in this notebook.

Three things the cell sets up for you:

- **Two models.** Part A uses a current model (`gpt-5-mini`), which can read pictures.
  Part B asks every question to that model **and** to an older one released in 2024
  (`gpt-4o-mini`). Everyone uses the same two, so results can be compared.
- **A cache.** Every answer is saved in a folder called `llm_cache/`. If you run a cell
  again, the saved answer is read back instead of asking again. Re-running is free and
  gives the same answer. (In Colab the folder is deleted when the session ends; the next
  session simply asks again.)
- **A budget.** A full run of this lab asks about 610 questions. The notebook stops itself
  at 900, which only happens if a loop runs away. If you share a key with a class, please do not
  raise the budget.

In [ ]:
# =====================================================================
# Lab 10 -- API key, the two models and the cache. Run this cell once.
# =====================================================================

api_key = getpass.getpass("Paste your API key, then press Enter: ").strip()

if not api_key:
    raise ValueError("No key entered. Re-run this cell and paste your key.")

# Two kinds of key work. An OpenAI key (starts with "sk-") talks to OpenAI directly.
# Any other key is a UT AI Studio (Portkey) key -- e.g. a class key -- which reaches
# the same OpenAI models through UT's Portkey gateway.
if api_key.startswith("sk-"):
    client = OpenAI(api_key=api_key, max_retries=10)
    MODEL_PREFIX = ""
    print(f"OpenAI key received ({len(api_key)} characters). It lives only in this session.")
else:
    client = OpenAI(api_key=api_key, base_url="https://api.portkey.ai/v1", max_retries=10)
    MODEL_PREFIX = "@grg356-spatiald-53312c/"     # the course workspace in UT AI Studio
    print(f"UT AI Studio key received ({len(api_key)} characters). It lives only in this session.")

MODEL_OLD = "gpt-4o-mini"                     # an ordinary model, released July 2024
MODEL_NEW = "gpt-5-mini"                      # a current model; it can also read pictures
DEFAULT_SYSTEM = "You are a helpful assistant."

CACHE_DIR = pathlib.Path("llm_cache")
CACHE_DIR.mkdir(exist_ok=True)

CALL_BUDGET = 900          # a clean full run is about 610
_calls_made = 0
_count_lock = threading.Lock()


def _image_part(path, max_side=1024):
    """Shrink a picture and turn it into the form the API accepts."""
    picture = Image.open(path).convert("RGB")
    picture.thumbnail((max_side, max_side))
    buffer = io.BytesIO()
    picture.save(buffer, format="JPEG", quality=85)
    data = base64.b64encode(buffer.getvalue()).decode()
    return {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + data}}


def ask_llm(question, system=DEFAULT_SYSTEM, model=MODEL_NEW, images=(), json_answer=False, attempt=1):
    """Ask one model one question and return its answer as text.

    system       the system prompt: an instruction that sets how the model behaves
    images       picture files to send along with the question (Part A)
    json_answer  True makes the model reply with a JSON object and nothing else
    attempt      change it (2, 3, ...) to ask again instead of reading the saved answer

    An answer you already got is read back from llm_cache/ instead of asked again.
    """
    global _calls_made
    image_ids = [hashlib.md5(pathlib.Path(p).read_bytes()).hexdigest() for p in images]
    name = hashlib.md5(json.dumps([model, system, question, image_ids, json_answer, attempt]).encode()).hexdigest()
    path = CACHE_DIR / (name + ".json")
    if path.exists():                                   # asked before -- free
        return json.loads(path.read_text(encoding="utf-8"))["answer"]

    with _count_lock:
        if _calls_made >= CALL_BUDGET:
            raise RuntimeError(
                f"STOPPED after {_calls_made} new questions -- over the {CALL_BUDGET} budget "
                "for Lab 10.\nA clean run needs about 610, so a loop has probably run away. "
                "Check your code, restart the kernel and run again.\nIf you share a key with a class, "
                "please do not raise CALL_BUDGET -- the whole class shares it."
            )
        _calls_made += 1

    content = question
    if images:
        content = [{"type": "text", "text": question}] + [_image_part(p) for p in images]
    extra = {"response_format": {"type": "json_object"}} if json_answer else {}
    reply = client.chat.completions.create(
        model=MODEL_PREFIX + model,
        messages=[
            {"role": "system", "content": system},     # how to behave
            {"role": "user",   "content": content},    # what to answer
        ],
        **extra,
    )
    answer = reply.choices[0].message.content or ""
    path.write_text(json.dumps({"model": model, "system": system, "question": question,
                                "images": [str(p) for p in images], "answer": answer}), encoding="utf-8")
    return answer


def ask_many(questions, system=DEFAULT_SYSTEM, model=MODEL_NEW, attempt=1):
    """Ask a list of questions, eight at a time. Returns the answers in the same order.

    Prints a progress line every 25 answers, so a long run never looks frozen.
    """
    answers = [None] * len(questions)
    done = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        jobs = {pool.submit(ask_llm, q, system, model, attempt=attempt): k for k, q in enumerate(questions)}
        for job in as_completed(jobs):
            answers[jobs[job]] = job.result()
            done += 1
            if done % 25 == 0 or done == len(questions):
                print(f"   {model.split('/')[-1]}: {done} of {len(questions)} answered", flush=True)
    return answers


def calls_used():
    """How much of the budget this session has used."""
    print(f"New questions sent this session: {_calls_made} of {CALL_BUDGET}")
    print("(answers read back from llm_cache/ are free and not counted)")


print(f"Models: {MODEL_OLD} (2024) and {MODEL_NEW} (current).  Answers are saved in {CACHE_DIR}/.")

# Part A -- AI as a cartographer

A web map has two halves: the **data** (where the streets, buildings and rivers are) and
the **style** (what colour, width and font each of them is drawn in). In Mapbox, the
style is a single text file in JSON format. Because a style is text, an LLM can write one.

**CartoAgent**, developed in Prof. Kang's GISense Lab, does exactly that. It splits the job
among several "agents", each one a prompt to a multimodal LLM:

1. **describe** a picture -- its content, colours and mood,
2. **design** a style sheet -- one colour, width and opacity per kind of map element,
3. **write** the full Mapbox style file,
4. **evaluate** the finished map against the picture, and suggest revisions.

The next cell shows results from the CartoAgent paper: photographs, paintings,
AI-generated maps and web maps (left), each turned into map styles at three scales.

In [ ]:
# Written for you -- results from the CartoAgent paper.
display(Image.open("data/cartoagent_examples.jpg"))
print("Source: CartoAgent (IJGIS, 2025), GISense Lab.")

## A1 -- A map style is a text file

`data/neighborhood_style.json` is the style we start from: a quiet, pale map of the UT
campus. It has **20 layers**, drawn bottom to top, of four kinds:

| kind | what it draws | what the style can set |
|---|---|---|
| `background` | everything under the map | `background-color` |
| `fill` | areas: parks, water, buildings | `fill-color`, `fill-opacity`, `fill-outline-color` |
| `line` | roads, paths, rail | `line-color`, `line-width`, `line-opacity` |
| `symbol` | text labels | `text-color`, `text-halo-color`, `text-halo-width`, `text-halo-blur` |

The cell below lists the layers with their current colours and shows the map they make.

In [ ]:
# Written for you -- read the starting style and show what it looks like.
base_style = json.loads(pathlib.Path("data/neighborhood_style.json").read_text())


def colour_of(layer):
    """The main colour of one layer, whichever kind of layer it is."""
    paint = layer.get("paint", {})
    for key in ("background-color", "fill-color", "line-color", "text-color"):
        if key in paint:
            return paint[key]
    return ""


print(pd.DataFrame([{"layer": l["id"], "kind": l["type"], "colour": colour_of(l)}
                    for l in base_style["layers"]]).to_string(index=False))
display(Image.open("data/base_map.jpg"))
print("The starting style over the UT campus.  Map data © Mapbox © OpenStreetMap.")

## A2 -- The four prompts

These are the CartoAgent prompts, lightly edited for this lab. **Read them before you run
anything.** Notice how much they spell out: the exact list of map elements, what may be
changed for each kind, the colour format, and the exact JSON shape of the answer. A model
can only fill in a file reliably when the prompt is this specific.

In [ ]:
# =====================================================================
# The four CartoAgent prompts. Read them -- you do not need to change them.
# =====================================================================

MAP_ELEMENTS = """
- background elements: Background.
- fill elements: Commercial_area, Green_space, Water, Commercial_building, Residential_building.
- line elements: Walkway, Street, Tertiary_road, Secondary_road, Primary_road, Metro.
- symbol_label elements: Water_label, Settlement_label, Street_label, Tertiary_road_label, Secondary_road_label, Primary_road_label, Natural_landscape_label, Cultural_landscape_label.
"""

ALLOWED_CHANGES = """
- For each background element, you can set the background-color.
- For each fill element, you can set the fill-outline-color, fill-opacity, fill-color.
- For each line element, you can set the line-color, line-width, line-opacity.
- For each symbol_label element, you can set the text-halo-width, text-halo-color, text-color, text-halo-blur.
"""

STYLE_SHEET_JSON = """
{
  "Explanation": "a few sentences on why you chose these styles",
  "background": {"Background": {"background-color": "#rrggbb"}},
  "fill": {
    "Commercial_area": {"fill-opacity": 0.0, "fill-color": "#rrggbb", "fill-outline-color": "#rrggbb"},
    ... the same three keys for Green_space, Water, Commercial_building, Residential_building
  },
  "line": {
    "Walkway": {"line-opacity": 0.0, "line-color": "#rrggbb", "line-width": 0.0},
    ... the same three keys for Street, Tertiary_road, Secondary_road, Primary_road, Metro
  },
  "symbol_label": {
    "Water_label": {"text-color": "#rrggbb", "text-halo-color": "#rrggbb", "text-halo-width": 0.0, "text-halo-blur": 0.0},
    ... the same four keys for every other symbol_label element
  }
}
"""

# 1 -- describe the picture
PROMPT_DESCRIBE = """
I want you to act as an image appreciator. I will provide you with an image. Your task is to
describe it in as much detail as possible, covering these topics:
- Content: What is in the image? What are the attributes and relationships of these things?
  What temporal and spatial information does the image reflect?
- Color: What is the color of the objects and the background? Give hexadecimal color values
  as accurately as possible, e.g. #000000. What effect does the combination of these colors
  give to the brightness and contrast of the image?
- Theme & Design: If the image is a painting, what is its theme (e.g. realistic, surreal)?
  What are the characteristics of the design (e.g. simple, detailed, complex)?
The image follows.
"""

# 2 -- design a style sheet from the description
PROMPT_STYLE_SHEET = """
I want you to act as a style sheet designer. I will provide you with a description of a
reference image, covering its content, color, theme and design. Your task is to design a
style sheet for the following map elements so that the rendered map is as consistent as
possible with the reference image.
[Data name]: UT Campus, Austin, Texas
[Map elements]:""" + MAP_ELEMENTS + """[What you can set]:""" + ALLOWED_CHANGES + """
Requirements:
1. Completeness: set the styles for all elements.
2. Unity: related elements, such as roads of different levels, should have similar styles;
   keep a clear contrast and visual hierarchy.
3. Rationality: take the real-world meaning of each element into account.
4. Explainability: explain the reasons for your choices in the "Explanation" field.
5. Normativeness: give all colors in hexadecimal format, e.g. #000000; give opacity as a
   decimal between 0 (transparent) and 1 (opaque).
Return only a JSON object in exactly this shape:""" + STYLE_SHEET_JSON + """
The description of the reference image follows.
"""

# 3 -- write the full Mapbox style file
PROMPT_WRITE_STYLE = """
I want you to act as a senior computer expert. I will provide you with a Mapbox style
specification file and a new style sheet. Your task is to modify the style specification
file according to the new style sheet while strictly following the original file's format.
Return the entire modified style specification file, including the parts you did not change.
When you meet a filtering expression, do not change the filtering condition; change only the
value. For example, "fill-opacity": ["match", ["get", "class"], ["commercial_area"], 1, 0]
gives features of class "commercial_area" an opacity of 1 and all others 0. To change that
opacity to 0.8, write ["match", ["get", "class"], ["commercial_area"], 0.8, 0].
Return only the JSON object of the modified style file. The original style file and the new
style sheet follow.
"""

# 4 -- evaluate the finished map against the picture
PROMPT_EVALUATE = """
I want you to act as a map evaluation expert. I will provide you with two images: first a
reference image, then a map designed in the style of the reference image. Evaluate whether
the style of the map is consistent with the style of the reference image, considering color,
theme and design together. Then make one of two choices: Accept or Revision.
Our map elements are:""" + MAP_ELEMENTS + """When revising, you can change:""" + ALLOWED_CHANGES + """
Requirements:
1. You do not need to modify all map elements.
2. Give all colors in hexadecimal format, e.g. #000000, and opacity as a decimal between 0 and 1.
3. Return only a JSON object.
If you choose Accept, return:
{"Action": "Accept", "Reason": "one or two sentences"}
If you choose Revision, return:
{"Action": "Revision", "Reason": "one or two sentences",
 "Modified style sheet": {the same shape as the style sheet, containing only the elements you change}}
The reference image and then the map follow.
"""

print("Prompt lengths (characters):",
      {name: len(p) for name, p in [("describe", PROMPT_DESCRIBE), ("style sheet", PROMPT_STYLE_SHEET),
                                     ("write style", PROMPT_WRITE_STYLE), ("evaluate", PROMPT_EVALUATE)]})

The next cell is written for you. It turns the four prompts into four functions, plus one
that runs the first three in a row:

| function | prompt | gives back |
|---|---|---|
| `describe_image(picture)` | 1 | a description, as text |
| `design_style_sheet(description)` | 2 | a style sheet (a dictionary) |
| `write_mapbox_style(style_sheet, out_file)` | 3 | a full Mapbox style, saved to `out_file` |
| `evaluate_map(picture, screenshot)` | 4 | Accept or Revision, with the model's reason |
| `style_from_image(picture, out_file)` | 1-3 | runs the first three for you |

One safety net: in `write_mapbox_style`, only the **colours, widths and opacities** of each
layer are taken from the model's reply. The rest of the file (which data each layer draws,
the filters, the fonts) is copied from the starting style, and a colour that is not a valid
hex code is replaced by the original one. Without this, one small slip by the model can make
Mapbox Studio refuse the whole file, or paint every building as a park.

In [ ]:
# =====================================================================
# Written for you -- the four CartoAgent steps as functions.
# =====================================================================

HEX = re.compile(r"^#([0-9a-fA-F]{3}|[0-9a-fA-F]{6}|[0-9a-fA-F]{8})$")


def describe_image(picture):
    """Prompt 1: the model describes the picture's content, colours and mood."""
    return ask_llm(PROMPT_DESCRIBE, system="You are an image appreciation expert.",
                   model=MODEL_NEW, images=[picture])


def design_style_sheet(description):
    """Prompt 2: turn the description into one style per map element."""
    answer = ask_llm(PROMPT_STYLE_SHEET + "\n" + description,
                     system="You are a Mapbox style sheet designer.",
                     model=MODEL_NEW, json_answer=True)
    return json.loads(answer)


def _keep_filter(original, value):
    """Keep the filter hidden inside an opacity such as ["match", ["get", "class"], ["park"], 1, 0].

    That expression means "1 for parks, 0 for everything else". If the model replies with a
    plain number, only the 1 is replaced; otherwise every feature would be drawn.
    """
    is_match = isinstance(original, list) and len(original) == 5 and original[0] == "match"
    if not is_match:
        return value
    if isinstance(value, (int, float)):
        return original[:3] + [value, original[4]]
    if isinstance(value, list) and len(value) == 5 and value[:3] == original[:3]:
        return value
    return original                                         # anything else: keep the original


def write_mapbox_style(style_sheet, out_file, attempt=1):
    """Prompt 3: the model edits the full style file. Saves it to out_file."""
    question = (PROMPT_WRITE_STYLE
                + "\nOriginal style specification file:\n" + json.dumps(base_style)
                + "\nNew style sheet:\n" + json.dumps(style_sheet))
    answer = ask_llm(question, system="You are a Mapbox style configuration expert.",
                     model=MODEL_NEW, json_answer=True, attempt=attempt)
    from_model = {l.get("id"): l for l in json.loads(answer).get("layers", [])}

    new_style = json.loads(json.dumps(base_style))         # a fresh copy of the starting style
    new_style["name"] = "Lab 10 -- " + pathlib.Path(out_file).stem
    changed, kept = 0, 0
    for layer in new_style["layers"]:
        reply_layer = from_model.get(layer["id"], {})
        for key, value in reply_layer.get("paint", {}).items():
            if key not in layer.get("paint", {}):
                continue                                    # only properties the layer already has
            if key.endswith("-color") and isinstance(value, str) and not HEX.match(value):
                kept += 1                                   # not a hex colour: keep the original
                continue
            value = _keep_filter(layer["paint"][key], value)
            if value != layer["paint"][key]:
                layer["paint"][key] = value
                changed += 1
    pathlib.Path(out_file).write_text(json.dumps(new_style, indent=2))
    print(f"Saved {out_file}: {changed} style values changed by the model"
          + (f", {kept} invalid colours kept as they were." if kept else "."))
    if changed == 0:
        print("The model changed nothing. Run again with attempt=2 to ask for a fresh answer.")
    return new_style


def evaluate_map(picture, screenshot):
    """Prompt 4: the model compares its map with the picture: Accept or Revision."""
    if not pathlib.Path(screenshot).exists():
        raise FileNotFoundError(
            f"No file called {screenshot} next to this notebook. Take the screenshot in "
            "Mapbox Studio (section A4), save it with exactly this name, and run the cell again.")
    answer = ask_llm(PROMPT_EVALUATE, system="You are a professional map evaluator.",
                     model=MODEL_NEW, images=[picture, screenshot], json_answer=True)
    return json.loads(answer)


def style_from_image(picture, out_file):
    """Prompts 1-3 in a row: picture -> description -> style sheet -> style file."""
    description = describe_image(picture)
    style_sheet = design_style_sheet(description)
    write_mapbox_style(style_sheet, out_file)
    return description, style_sheet


def show_style_sheet(style_sheet):
    """A readable table of the colours the model chose."""
    rows = []
    for kind in ("background", "fill", "line", "symbol_label"):
        for element, values in style_sheet.get(kind, {}).items():
            colour = next((v for k, v in values.items() if k in
                           ("background-color", "fill-color", "line-color", "text-color")), "")
            rows.append({"kind": kind, "element": element, "colour": colour})
    print(pd.DataFrame(rows).to_string(index=False))

## A3 -- From a photograph to a map style

Our first picture is a photograph of a misty mountain forest, `data/inspiration.jpg`. The
next cell shows it, then runs prompts 1, 2 and 3. It takes a minute or two the first time.

In [ ]:
# Written for you -- picture -> description -> style sheet -> style file.
display(Image.open("data/inspiration.jpg").resize((640, 480)))

description = describe_image("data/inspiration.jpg")
print("=== 1. The model's description ===\n")
print(description)

style_sheet = design_style_sheet(description)
print("\n=== 2. The style sheet it designed ===\n")
show_style_sheet(style_sheet)
print("\nWhy:", style_sheet.get("Explanation", "(no explanation given)"))

print("\n=== 3. The full style file ===\n")
write_mapbox_style(style_sheet, "modified_style.json")

## A4 -- See your map in Mapbox Studio

Mapbox Studio is Mapbox's online map designer. You need a free account:

1. Go to https://www.mapbox.com, click **Sign up**, and use your UT email. Choose to start
   **without a credit card**. Nothing can be charged to an account with no card on file.
2. Download `modified_style.json` from this notebook's folder (in Colab: the **Files** panel
   on the left, right-click the file, **Download**).
3. Open https://studio.mapbox.com and click **New style**. **Upload is not one of the
   template tiles** -- scroll past them to **Other options**, choose **Upload**, and select
   `modified_style.json`. Your map opens in the editor. (Already made a style? Its **⋯**
   menu has **Replace**, which does the same thing.)
4. Do the same with `data/neighborhood_style.json`, the style we started from, so you can
   compare the two side by side.
5. Take a screenshot of **just the map area** of your new style, and save it as
   **`my_map.png`** next to this notebook (in Colab: drag it into the **Files** panel).

The next cell needs `my_map.png`.

## A5 -- Let the model judge its own map

Prompt 4 shows the model the photograph and your screenshot, and asks: does the map match
the picture? It answers **Accept** or **Revision**. With **Revision** it also returns new
values for the elements it wants to change, and the cell turns them into
`revised_style.json`, which you can upload to Studio the same way.

In [ ]:
# Written for you -- the model evaluates the map it designed.
evaluation = evaluate_map("data/inspiration.jpg", "my_map.png")
display(Image.open("my_map.png"))      # your screenshot, so it is part of the notebook you hand in
print("Action:", evaluation.get("Action"))
print("Reason:", evaluation.get("Reason", "(no reason given)"))

if evaluation.get("Action") == "Revision":
    # Fold the suggested changes into a copy of the first style sheet. The model does not
    # always answer in the same shape: it may say "fill elements" where the sheet says "fill",
    # and "background" can be one level shallower ({"background-color": ...}). Handle both.
    revised_sheet = json.loads(json.dumps(style_sheet))
    names = {k.lower(): k for k in revised_sheet}
    for kind, elements in evaluation.get("Modified style sheet", {}).items():
        if not isinstance(elements, dict):
            continue
        kind = names.get(kind.lower().replace(" elements", "").strip(), kind)
        if not isinstance(revised_sheet.get(kind), dict):
            revised_sheet[kind] = {}
        for element, values in elements.items():
            if isinstance(values, dict):              # element -> {property: value}
                revised_sheet[kind].setdefault(element, {}).update(values)
            else:                                     # property -> value, e.g. background
                revised_sheet[kind][element] = values
    write_mapbox_style(revised_sheet, "revised_style.json")
    print("Upload revised_style.json to Studio to see the revision.")

# Exercise 1 — A Map in the Style of an Artwork

**What you do:** choose an artwork, let the model design a map from it, and judge the result.<br>
**Hand in:** your artwork link, its style sheet, your map screenshot with the model's verdict,
and 2-3 sentences.

1. Choose a work of art you like from the **Blanton Museum of Art**'s collection at UT:
   https://collection.blantonmuseum.org . If you cannot download an image there, any artwork
   from an open-access museum collection works, for example The Met's open-access images.
2. Save the image next to this notebook as **`my_artwork.jpg`**, and paste the link to the
   artwork into `ARTWORK_LINK` below.
3. Run the first cell: it writes **`my_artwork_style.json`**.
4. Upload that file to Mapbox Studio (as in A4), take a screenshot of the map, and save it
   as **`my_artwork_map.png`**.
5. Run the second cell: the model evaluates your map.

In [ ]:
# ============================================================
# EXERCISE 1, part 1 -- paste the link to your artwork, then run this cell.
# ============================================================

ARTWORK_LINK = ""        # <-- the web page of your artwork

if not ARTWORK_LINK.strip():
    raise NotImplementedError("Paste the link to your artwork into ARTWORK_LINK, then run this cell again.")

display(Image.open("my_artwork.jpg"))
art_description, art_sheet = style_from_image("my_artwork.jpg", "my_artwork_style.json")
show_style_sheet(art_sheet)
print("\nWhy:", art_sheet.get("Explanation", "(no explanation given)"))

In [ ]:
# EXERCISE 1, part 2 -- after saving my_artwork_map.png, run this cell.
art_evaluation = evaluate_map("my_artwork.jpg", "my_artwork_map.png")
display(Image.open("my_artwork_map.png"))
print("Action:", art_evaluation.get("Action"))
print("Reason:", art_evaluation.get("Reason", "(no reason given)"))

### Questions (Exercise 1) -- 2-3 sentences in total

1. Does your map look like your artwork? Name one thing the model got right and one thing
   it got wrong.
2. The model judged its own map. Do you agree with its verdict? Why or why not?

### Your answers (Exercise 1)

*Double-click this cell and write your 2-3 sentences here.*

# Part B -- Can we trust what AI knows about places?

In Part A the model designed a map and then judged its own work. In Part B we check a
model's work against an **outside** source: official data on all 254 Texas counties.

## B1 -- Asking an LLM about a place

Every question has two parts:

- the **system prompt** sets how the model behaves ("You are a helpful assistant"),
- the **question** is what you want answered.

`ask_llm(question, model=...)` sends both to one model and returns the answer as text.
Below we ask both models the same question about two counties: Travis County (Austin,
about 1.3 million people) and Loving County (48 people, the smallest county in the United
States by population).

In [ ]:
# Written for you -- the same question about a big county and a tiny one, to both models.
for county in ["Travis", "Loving"]:
    question = f"In two sentences, what is {county} County, Texas known for?"
    print(f"Q: {question}")
    for model in [MODEL_OLD, MODEL_NEW]:
        print(f"   {model}: {ask_llm(question, model=model)}")
    print()

All four answers sound confident. Are the Loving County answers right? You probably cannot
tell without looking it up, and neither can most people who ask. To measure how good the
answers are, we need a question with an answer we can check. That is section B2.

## B2 -- The neighbor test

**Question:** *Which Texas counties share a border with this county?*

This question has an official answer. The U.S. Census Bureau publishes a list of which
counties are next to each other. It is in `data/tx_county_neighbors.csv`, one row per pair:

| column | meaning |
|---|---|
| `county`, `neighbor` | two Texas counties that are next to each other |
| `touch` | `edge` if they share a stretch of border, `corner` if they only meet at a point |

Neighbors in other states are left out, because we only ask about Texas counties.

`data/tx_counties.geojson` has the county shapes and each county's 2024 population (Census
Bureau estimate).

### How an answer is scored

For each county, the answer is compared with the real list:

- **correct** -- a name the model gave that really is a neighbor (a corner neighbor
  counts as correct, but you are not penalized for leaving it out)
- **wrong** -- a name that is not a neighbor, or is not a Texas county at all
- **missed** -- a real neighbor (sharing an edge) that the model did not name

From these, three numbers between 0 and 1:

- **precision** -- of the names it gave, the share that are correct
- **recall** -- of the real neighbors, the share it named
- **score** -- one number combining the two. It is high only when the list is both
  correct **and** complete. (It is the *F1 score*: $\text{score} = \frac{2 \cdot
  \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$.)

In [ ]:
# Written for you -- load the counties and the official list of neighbors.
counties  = gpd.read_file("data/tx_counties.geojson")
neighbors = pd.read_csv("data/tx_county_neighbors.csv")

# For each county: the neighbors that share an edge, and the ones that only touch a corner.
EDGE   = {c: set(g.loc[g["touch"] == "edge",   "neighbor"]) for c, g in neighbors.groupby("county")}
CORNER = {c: set(g.loc[g["touch"] == "corner", "neighbor"]) for c, g in neighbors.groupby("county")}

print(f"{len(counties)} counties, {len(neighbors) // 2} neighbor pairs")
print(f"Population: {counties['population'].min():,} (smallest) to "
      f"{counties['population'].max():,} (largest)\n")
print("Travis County's neighbors:", sorted(EDGE["Travis"]))

The next cell is written for you. It defines the question and three functions:

- `score_answer(county, answer)` -- reads the county names out of a reply and scores them,
- `run_neighbor_test(names, model=...)` -- asks one model the question for a list of
  counties and returns a table with one row per county,
- `show_county(county)` -- prints the question, both models' answers and their scores for
  one county.

In [ ]:
# =====================================================================
# Written for you -- the neighbor question and how its answers are scored.
# =====================================================================

NEIGHBOR_QUESTION = (
    "Which Texas counties share a border with {county} County, Texas? "
    "List only Texas counties. Reply with the county names only, one per line, "
    "without the word 'County'."
)


def _key(name):
    """'De Witt', 'DeWitt County' -> 'dewitt', so spelling differences still match."""
    return re.sub(r"[^a-z]", "", name.lower())


NAME_OF = {_key(c): c for c in counties["county"]}       # every real Texas county name
POPULATION = dict(zip(counties["county"], counties["population"]))


def parse_counties(answer):
    """Pull county names out of a reply, one per line or separated by commas."""
    names = []
    for line in answer.splitlines():
        line = line.split(":")[-1]                     # "...the following counties: Hays, ..."
        for piece in re.split(r"[,;]|\band\b", line):
            piece = re.sub(r"\(.*?\)", "", piece)                       # "(to the south)"
            piece = re.sub(r"^\s*(\d+[.)]|[-*•])\s*", "", piece)   # list markers: 1.  -  *
            piece = re.sub(r"\b(county|counties|texas|tx)\b", "", piece, flags=re.I)
            piece = piece.strip(" .*")
            if not piece or piece[0].islower() or len(piece.split()) > 3:
                continue                               # part of a sentence, not a name
            if _key(piece) not in {_key(n) for n in names}:
                names.append(piece)
    return names


def score_answer(county, answer):
    """Compare the model's list of neighbors with the official one."""
    named   = parse_counties(answer)
    real    = {NAME_OF[_key(n)] for n in named if _key(n) in NAME_OF}
    edge    = EDGE[county]
    allowed = edge | CORNER.get(county, set())

    correct = real & allowed
    wrong   = sorted(real - allowed) + [n for n in named if _key(n) not in NAME_OF]
    missed  = edge - real

    precision = len(correct) / len(named) if named else 0.0
    recall    = len(edge & real) / len(edge)
    score     = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"county": county, "population": POPULATION[county],
            "n_named": len(named), "correct": sorted(correct), "wrong": wrong,
            "missed": sorted(missed), "precision": precision, "recall": recall,
            "score": score}


def run_neighbor_test(names, system=DEFAULT_SYSTEM, model=MODEL_NEW, attempt=1):
    """Ask one model the neighbor question for every county in `names`; one row per county."""
    answers = ask_many([NEIGHBOR_QUESTION.format(county=c) for c in names], system, model, attempt)
    return pd.DataFrame([score_answer(c, a) for c, a in zip(names, answers)])


def show_county(county, system=DEFAULT_SYSTEM):
    """Print the question, both models' answers and how each was scored, for one county."""
    question = NEIGHBOR_QUESTION.format(county=county)
    print(f"=== {county} County  (population {POPULATION[county]:,}) ===")
    print(f"Real neighbors:  {sorted(EDGE[county])}")
    if CORNER.get(county):
        print(f"Corner only:     {sorted(CORNER[county])}")
    for model in [MODEL_OLD, MODEL_NEW]:
        answer = ask_llm(question, system, model)
        s = score_answer(county, answer)
        print(f"\n--- {model} ---")
        print("   " + (answer.strip() or "(empty answer)").replace("\n", "\n   "))
        print(f"correct {s['correct']}\nwrong   {s['wrong']}\nmissed  {s['missed']}")
        print(f"precision {s['precision']:.2f}   recall {s['recall']:.2f}   score {s['score']:.2f}")
    print()

Here is one county, so you can see what each model said and how it was scored.

In [ ]:
show_county("Travis")

### Ask about all 254 counties

The next cell asks both models the neighbor question for every Texas county, eight
questions at a time -- about 500 questions in all.

**The first run takes about 5-10 minutes**, mostly because the current model thinks before
it answers. A progress line appears every 25 answers (`gpt-5-mini: 125 of 254 answered`),
so you can see it is working.

- **Do not stop the cell or run it a second time while it is going.** It is not frozen.
- If it does get interrupted (wifi drops, laptop sleeps), just run the cell again. Every
  answer already received is saved in `llm_cache/`, so it picks up where it stopped and
  those answers cost nothing.
- After the first full run, this cell finishes in seconds.

In [ ]:
# Written for you -- the neighbor test for every Texas county, both models.
all_counties = sorted(counties["county"])
results_old = run_neighbor_test(all_counties, model=MODEL_OLD)
results_new = run_neighbor_test(all_counties, model=MODEL_NEW)

print("Average over 254 counties:")
print(pd.DataFrame({MODEL_OLD: results_old[["precision", "recall", "score"]].mean(),
                    MODEL_NEW: results_new[["precision", "recall", "score"]].mean()}).round(2), "\n")
calls_used()

# the 10 counties the current model knows worst, with the 2024 model's score beside them
lowest = results_new.sort_values("score").head(10)[["county", "population", "score"]]
lowest = lowest.rename(columns={"score": f"score {MODEL_NEW}"})
lowest[f"score {MODEL_OLD}"] = lowest["county"].map(results_old.set_index("county")["score"])
lowest.round(2)

### Does the score depend on population? Did the newer model fix it?

The next cell draws:

1. a **map** of each county's score, one map per model,
2. a **scatter plot** of score against population for both models, with population on a
   log scale (each step to the right is ten times more people).

It also splits the counties into four equal groups by population and prints each model's
average score in each group, and each model's **rank correlation** between population and
score. The rank correlation runs from -1 to 1. Near 0 means no relationship; positive
means bigger counties tend to score higher.

In [ ]:
# Written for you -- maps, scatter plot and summary by population group, for both models.
runs = {MODEL_OLD: results_old, MODEL_NEW: results_new}

fig, ax = plt.subplot_mosaic([["old", "new"], ["scatter", "scatter"]], figsize=(14, 12))
for key, (model, res) in zip(["old", "new"], runs.items()):
    scored = counties.merge(res[["county", "score"]], on="county").to_crs(3083)
    scored.plot(column="score", cmap="RdYlGn", vmin=0, vmax=1, legend=True,
                edgecolor="white", linewidth=0.3, ax=ax[key],
                legend_kwds={"label": "score (0 = wrong, 1 = perfect)", "shrink": 0.6})
    ax[key].set_title(f"Neighbor test score: {model}")
    ax[key].set_axis_off()

for model, res in runs.items():
    ax["scatter"].scatter(res["population"], res["score"], s=14, alpha=0.5, label=model)
ax["scatter"].set_xscale("log")
ax["scatter"].set_xlabel("County population, 2024 (log scale)")
ax["scatter"].set_ylabel("score")
ax["scatter"].set_ylim(-0.05, 1.05)
ax["scatter"].set_title("Score vs. population")
ax["scatter"].legend()
plt.tight_layout()
plt.show()

groups = pd.qcut(counties.set_index("county")["population"], 4,
                 labels=["smallest 25%", "second 25%", "third 25%", "largest 25%"])
by_group = pd.DataFrame({model: res.set_index("county")["score"].groupby(groups, observed=True).mean()
                         for model, res in runs.items()}).round(2)
print("Average score by population group:")
print(by_group, "\n")

for model, res in runs.items():
    rank_corr = res["population"].rank().corr(res["score"].rank())
    print(f"Rank correlation between population and score, {model}: {rank_corr:.2f}")

### A score of 0.00 is not always a wrong answer

Before you read a **0.00** as "the model knows nothing about this county", open one with
`show_county()` and read the reply itself.

A score of 0.00 can come from two different kinds of reply: a list of the **wrong** counties,
or **no list at all** -- for example a reply that asks you a question instead of answering.
The model cannot look anything up: we send one question and read one reply, with no web
search and no second turn.

Both are scored 0.00, because someone who needed the neighbor list did not get one. But they
are different failures. A wrong county name can be checked against a map; a reply that sounds
like work in progress can be mistaken for progress. Look at which model gives which kind of
reply -- you will need it for Exercise 2.


# Exercise 2 — Where the Models Go Wrong

**What you do:** choose two counties and read what both models said about them.<br>
**Hand in:** `show_county()` output for your two counties, and 3-4 sentences.

Look at the maps, the scatter plot and the tables above. Then choose two counties in the
cell below:

- **one of the ten counties the current model scores lowest** -- the table printed after
  "Ask about all 254 counties" lists them;
- **one county you are curious about** -- your home county, a place you have been, or one
  where the 2024 model scored low.

Answer in **3-4 sentences in total**:

1. Do both models know small counties worse than large ones? Quote the rank correlations.
2. Did the newer model close the gap between the smallest and the largest counties, or did
   every county simply score a bit higher? Use the "smallest 25%" and "largest 25%" rows.
3. For each of your two counties, what kind of mistake did each model make? Say whether a
   model named counties that are not neighbors, missed real ones, or did not answer at all.

In [ ]:
# ============================================================
# EXERCISE 2 -- choose two counties, then run this cell.
# ============================================================
# ZERO_COUNTY : one of the ten counties the current model scores lowest. They
#               are listed in the table printed after "Ask about all 254 counties".
# MY_COUNTY   : any other Texas county you are curious about -- your home county,
#               somewhere you have been, or one where the 2024 model scored low.
# Write the name without the word "County", e.g. "Travis".

ZERO_COUNTY = ""
MY_COUNTY   = ""

if not ZERO_COUNTY.strip() or not MY_COUNTY.strip():
    raise NotImplementedError("Fill in ZERO_COUNTY and MY_COUNTY above, then run this cell again.")

_known = {c.lower(): c for c in counties["county"]}
ZERO_COUNTY, MY_COUNTY = (_known.get(n.strip().lower().removesuffix(" county"), n.strip())
                          for n in (ZERO_COUNTY, MY_COUNTY))
lowest_10 = list(results_new.sort_values(["score", "population"])["county"].head(10))
for n in (ZERO_COUNTY, MY_COUNTY):
    if n not in POPULATION:
        raise ValueError(f"'{n}' is not a Texas county name. Check the spelling, without the word County.")
if ZERO_COUNTY not in lowest_10:
    raise ValueError(f"ZERO_COUNTY should be one of the ten lowest-scoring counties: {', '.join(lowest_10)}")
if ZERO_COUNTY == MY_COUNTY:
    raise ValueError("Choose two different counties.")

for name in (ZERO_COUNTY, MY_COUNTY):
    show_county(name)

### Your answers (Exercise 2)

*Double-click this cell and write your 3-4 sentences here.*

## B3 -- Does a better prompt fix it?

Maybe the model only needs better instructions. In this section you write your own
**system prompt** and run the neighbor test again with the **current model** on the **30
least populous counties**, where scores are lowest. For example, you could:

- tell the model it is an expert on Texas geography,
- tell it to list only counties it is sure about, and to say so when it is not sure,
- ask it to think about which counties lie to the north, south, east and west.

The cell compares your prompt with the default one (`"You are a helpful assistant."`).
The default answers are already saved from section B2. The cell then asks the default
prompt a **second time** and asks your prompt, so 60 questions are new.

**Before you read too much into a small change:** an LLM does not give exactly the same
answer every time you ask. Ask the same question twice and the list can differ by a county
or two. With one run on 30 counties, a change of a few hundredths in the score may be that
run-to-run variation rather than your prompt. Only a clear change, or one that shows up
in both precision and recall the way you expected, is good evidence that the prompt worked.

**The cell measures that variation for you.** The column *default, asked again* is the same
prompt asked a second time. However far apart the two default columns are, that is how much
the score moves by chance alone -- your prompt has to beat it.

In [ ]:
# ============================================================
# EXERCISE 3 -- write your system prompt, then run this cell.
# ============================================================

MY_SYSTEM = ""      # <-- your system prompt, in quotes

if not MY_SYSTEM.strip():
    raise NotImplementedError("Write your system prompt in MY_SYSTEM, then run this cell again.")

smallest_30 = list(counties.sort_values("population")["county"].head(30))
default_30 = run_neighbor_test(smallest_30, model=MODEL_NEW)             # saved from section B2
again_30   = run_neighbor_test(smallest_30, model=MODEL_NEW, attempt=2)  # the SAME prompt, asked again: 30 new
mine_30    = run_neighbor_test(smallest_30, system=MY_SYSTEM, model=MODEL_NEW)  # your prompt: 30 new

compare = pd.DataFrame({
    "default prompt":       default_30[["precision", "recall", "score"]].mean(),
    "default, asked again": again_30[["precision", "recall", "score"]].mean(),
    "your prompt":          mine_30[["precision", "recall", "score"]].mean(),
}).round(2)
print(f"30 least populous counties, {MODEL_NEW} -- average over the 30:")
print(compare)
calls_used()

# Exercise 3 — Does a Better Prompt Fix It?

**What you do:** write your own system prompt in the cell above and run it.<br>
**Hand in:** your prompt, the comparison table, and 2-3 sentences.

Answer in **2-3 sentences in total**:

1. What system prompt did you write? Did the score change by more than the two default
   columns differ from each other -- the model's own run-to-run variation?
2. A planner in one of the low-scoring counties uses an LLM to draft a report on how far
   residents travel to a hospital. Name one thing that could go wrong, and one check you
   would do.

### Your answers (Exercise 3)

*Double-click this cell and write your 2-3 sentences here.*

# Deliverables

Turn in your `.ipynb` file with every cell run top to bottom, so all outputs are visible.
Your two screenshots appear inside it -- A5 and Exercise 1 display them -- so you do not
upload them separately.

**Part A**

1. A3 and A5: the description, style sheet and evaluation for the photograph, and the
   `my_map.png` screenshot.
2. Exercise 1: the link to your artwork, its style sheet, the `my_artwork_map.png`
   screenshot, the model's evaluation, and your 2-3 sentence answer.

**Part B**

3. B2: the two maps, the scatter plot, and the table of scores by population group.
4. Exercise 2: `show_county()` output for your two counties, and your 3-4 sentence answer.
5. Exercise 3: your system prompt, the comparison table, and your 2-3 sentence answer.